<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/supervised/05_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S5 ロジスティック回帰と分類(notebook)

本文:[`05-logistic-regression.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/supervised/05-logistic-regression.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-6):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

def make_blobs2(n=200, gap=1.0, seed=0):
    # クラス 0 とクラス 1 の点を、2つの正規分布から作る。gap を小さくすると重なりが増える
    rng = np.random.default_rng(seed)
    X0 = rng.normal([-gap, -0.5 * gap], 0.8, (n // 2, 2))
    X1 = rng.normal([gap, 0.8 * gap], 0.8, (n - n // 2, 2))
    X = np.vstack([X0, X1])
    t = np.concatenate([np.zeros(n // 2), np.ones(n - n // 2)])
    return X, t

def add_bias(X):
    return np.column_stack([np.ones(len(X)), X])           # S2 と同じ:先頭に 1 の列を足す

X_train, t_train = make_blobs2(200, seed=0)
X_test, t_test = make_blobs2(200, seed=1)
Phi_train, Phi_test = add_bias(X_train), add_bias(X_test)
plt.figure(figsize=(4, 4)); plt.scatter(*X_train.T, c=t_train, cmap="coolwarm", s=12); plt.axis("equal"); plt.title("training data"); plt.show()

## 1. シグモイド関数

本文の式 (1) の $\sigma(a) = 1 / (1 + e^{-a})$ を、あふれない形で書く(本文 §4 の WARNING)。`np.where` で場合分けし、`np.exp` には 0 以下の値だけを渡す。

**手計算**:本文の確認課題1を解く。

- 手計算:(ここに書く)

In [ ]:
def sigmoid(a):
    a = np.asarray(a, dtype=float)
    raise NotImplementedError("TODO: e = np.exp(-np.abs(a)) を使い、a >= 0 なら 1/(1+e)、a < 0 なら e/(1+e)")

In [ ]:
check("sigmoid(0), sigmoid(2), sigmoid(-2)", sigmoid(np.array([0.0, 2.0, -2.0])), [0.5, 1 / (1 + np.exp(-2)), 1 / (1 + np.exp(2))])
check("sigmoid(-a) = 1 - sigmoid(a)", sigmoid(-np.linspace(-5, 5, 11)), 1 - sigmoid(np.linspace(-5, 5, 11)))
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("error")                          # 警告が出たらエラーにする
    check("a = ±800 でも、あふれずに 0 と 1", sigmoid(np.array([-800.0, 800.0])), [0.0, 1.0])

## 2. 交差エントロピーと、その勾配

本文の式 (2) の損失は用意してある。式 (3) の勾配 $\frac{1}{N}\Phi^\top (y - t)$ を書き、数値微分で確かめる。

**手計算**:本文の確認課題3を解く。

- 手計算:(ここに書く)

In [ ]:
def cross_entropy(w, Phi, t):
    y = sigmoid(Phi @ w)
    eps = 1e-12                                             # log(0) を避ける
    return -np.mean(t * np.log(y + eps) + (1 - t) * np.log(1 - y + eps))

def grad_cross_entropy(w, Phi, t):
    raise NotImplementedError("TODO: Phi.T @ (sigmoid(Phi @ w) - t) / len(t)")

In [ ]:
Phi1, t1 = np.array([[1.0, 2.0]]), np.array([1.0])
check("確認課題3:損失は log 2", cross_entropy(np.zeros(2), Phi1, t1), np.log(2))
check("確認課題3:勾配は (-0.5, -1)", grad_cross_entropy(np.zeros(2), Phi1, t1), [-0.5, -1.0])
w0 = np.array([0.1, -0.2, 0.3]); h = 1e-6
num = np.array([(cross_entropy(w0 + e, Phi_train, t_train) - cross_entropy(w0 - e, Phi_train, t_train)) / (2 * h) for e in np.eye(3) * h])
check("式 (3) = 数値微分", grad_cross_entropy(w0, Phi_train, t_train), num, tol=1e-7)

## 3. 勾配法で学習する

本文の式 (4) をくり返す。

**予想**:学習率 0.5 で 500 回くり返すと、損失はどう変わるか(単調に減るか)。評価用のデータの正解率はどれくらいか。

- 予想:(ここに書く)

In [ ]:
def train_logistic(Phi, t, lr=0.5, n_iters=500, lam=0.0):
    w = np.zeros(Phi.shape[1])
    history = []
    for i in range(n_iters):
        g = grad_cross_entropy(w, Phi, t)
        g[1:] += lam * w[1:]                                # 正則化(§5 で使う。切片 w[0] には入れない)
        raise NotImplementedError("TODO: 勾配法の1ステップ(損失を小さくするので、勾配を引く)")
        history.append(cross_entropy(w, Phi, t))
    return w, np.array(history)

In [ ]:
w, history = train_logistic(Phi_train, t_train)
check("損失は単調に減る", np.all(np.diff(history) <= 1e-12), True)
check("勾配はほぼ 0(最小に着いた)", grad_cross_entropy(w, Phi_train, t_train), np.zeros(3), tol=1e-3)
accuracy = lambda w, Phi, t: np.mean((sigmoid(Phi @ w) > 0.5) == t)
print(f"w = {w.round(3)}   学習用の正解率 {accuracy(w, Phi_train, t_train):.3f}   評価用の正解率 {accuracy(w, Phi_test, t_test):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history); ax[0].set_xlabel("iteration"); ax[0].set_ylabel("cross entropy")
g1, g2 = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 4, 200))
prob = sigmoid(add_bias(np.column_stack([g1.ravel(), g2.ravel()])) @ w).reshape(g1.shape)
cs = ax[1].contourf(g1, g2, prob, levels=np.linspace(0, 1, 11), cmap="coolwarm", alpha=0.5)
ax[1].contour(g1, g2, prob, levels=[0.5], colors="k")
ax[1].scatter(*X_test.T, c=t_test, cmap="coolwarm", s=10, edgecolors="k", linewidths=0.2)
plt.colorbar(cs, ax=ax[1], label="p(t=1|x)"); ax[1].set_title("test data, boundary p = 0.5"); plt.tight_layout(); plt.show()

決定境界(黒い線)から離れるほど、確率は 0 または 1 に近づく(本文 §5)。下で、境界からの距離と確率を数値で確かめ、しきい値を変えたときの変化を見る。

In [ ]:
for x1 in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    print(f"x = ({x1:+.1f}, 0.0):p(t=1|x) = {sigmoid(np.array([1.0, x1, 0.0]) @ w):.3f}")
p_test = sigmoid(Phi_test @ w)
for th in [0.3, 0.5, 0.7]:
    pred = p_test > th
    missed = np.sum((t_test == 1) & ~pred)                  # 見逃し:本当はクラス 1 なのに 0 と判定
    false_alarm = np.sum((t_test == 0) & pred)              # 誤警報:本当はクラス 0 なのに 1 と判定
    print(f"しきい値 {th}:見逃し {missed:3d} 件、誤警報 {false_alarm:3d} 件")

## 4. 条件を変える

1. 学習率を 0.05、0.5、5、50 に変えると、損失の減り方はどうなるか
2. 2つのクラスを近づける(`gap=0.3`)と、正解率と確率の出方はどう変わるか

In [ ]:
for lr in [0.05, 0.5, 5.0, 50.0]:
    _, hist = train_logistic(Phi_train, t_train, lr=lr, n_iters=200)
    print(f"1. lr = {lr:5}: 損失 10回目 {hist[9]:.4f}、200回目 {hist[-1]:.4f}、途中で増えたか {bool(np.any(np.diff(hist) > 1e-12))}")
Xc, tc = make_blobs2(200, gap=0.3, seed=0); Xc_te, tc_te = make_blobs2(200, gap=0.3, seed=1)
wc, _ = train_logistic(add_bias(Xc), tc)
pc = sigmoid(add_bias(Xc_te) @ wc)
print(f"2. gap = 0.3:評価用の正解率 {accuracy(wc, add_bias(Xc_te), tc_te):.3f}、確率が 0.1 未満か 0.9 より大きい点の割合 {np.mean((pc < 0.1) | (pc > 0.9)):.2f}")
print(f"   gap = 1.0:評価用の正解率 {accuracy(w, Phi_test, t_test):.3f}、確率が 0.1 未満か 0.9 より大きい点の割合 {np.mean((p_test < 0.1) | (p_test > 0.9)):.2f}")

## 5. 完全に分けられるデータと正則化

クラスを遠ざけて(`gap=3`)、直線で完全に分けられるデータを作る。正則化なしと、あり( $\lambda = 0.01$)で、くり返しの回数を増やしたときの $\lVert w \rVert$ を比べる(本文 §6)。

**予想**:正則化なしで、くり返しを 10 倍にすると、 $\lVert w \rVert$ はどうなるか。

- 予想:(ここに書く)

In [ ]:
Xs_, ts_ = make_blobs2(200, gap=3.0, seed=0)
Phis_ = add_bias(Xs_)
for lam in [0.0, 0.01]:
    norms = [np.linalg.norm(train_logistic(Phis_, ts_, n_iters=n, lam=lam)[0]) for n in [100, 1000, 10000]]
    print(f"lambda = {lam}: |w| (100, 1000, 10000 回) = {np.round(norms, 2)}")
w_nr = train_logistic(Phis_, ts_, n_iters=10000)[0]
check("正則化なしでは、くり返すほど |w| が大きくなる", np.linalg.norm(w_nr) > np.linalg.norm(train_logistic(Phis_, ts_, n_iters=1000)[0]), True)

## 6. 研究で使うときは

`cpai_lab` には、分類の関数は入っていない(本文 §9)。研究で scikit-learn の `LogisticRegression` を使うときは、次の対応を覚えておく。

| この notebook | scikit-learn |
|---|---|
| `lam`(正則化の強さ) | `C`( $1/\lambda$ にあたる。小さいほど強い。初めから `C=1.0` で正則化が入っている) |
| `sigmoid(Phi @ w)` | `predict_proba(X)[:, 1]` |
| しきい値 0.5 で判定 | `predict(X)` |
| 勾配法 | もっと速い方法(準ニュートン法など)で、同じ損失を最小にする |

## 7. 説明(自分の言葉で書く)

1. ロジスティック回帰の出力は何を表しているか。決定境界は、出力がいくつの所か
   - (ここに書く)
2. 交差エントロピーの勾配が「予測と正解の差 × 入力」になることは、学習の動きとしてどういう意味か
   - (ここに書く)
3. 完全に分けられるデータで正則化が必要になるのはなぜか
   - (ここに書く)